Runtime → Change runtime type → **GPU** before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/xai-cnn-vit'
!mkdir -p $CKPT_DIR

In [ ]:
# Replace with your repo URL. For a private repo use a GitHub token.
REPO = 'https://github.com/hsfcoulibaly/Explanability-Methods.git'
!git clone -q $REPO /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo/ViT
!pip install -q timm captum

## Fine-tune (VIT)
Checkpoint saves to Drive every time test accuracy improves, so a disconnect doesn't lose it.

In [ ]:
!python -m src.train --arch vit --epochs 5 --lr 1e-4 --batch-size 128 --out $CKPT_DIR/vit.pt

## Check the checkpoint + Grad-CAM on a few images

In [ ]:
import torch, matplotlib.pyplot as plt
from src.models import load_checkpoint
from src.data import get_eval_subset, denormalize
from src.explain import gradcam, attention_rollout

device = 'cuda'
model, ckpt = load_checkpoint('vit', f'{CKPT_DIR}/vit.pt', device)
print('test acc:', ckpt['test_acc'], 'epoch:', ckpt['epoch'])

x, y = next(iter(get_eval_subset(batch_size=6)))
x = x.to(device)
cam, pred = gradcam(model, x)
maps = [('Grad-CAM', cam)]
roll, _ = attention_rollout(model, x)
maps.append(('Rollout', roll))

fig, ax = plt.subplots(1 + len(maps), 6, figsize=(15, 2.6 * (1 + len(maps))))
for i in range(6):
    ax[0, i].imshow(denormalize(x[i]).permute(1, 2, 0).cpu())
    ax[0, i].set_title(f'y={y[i].item()} pred={pred[i].item()}', fontsize=9)
    for r, (name, m) in enumerate(maps, 1):
        ax[r, i].imshow(denormalize(x[i]).permute(1, 2, 0).cpu())
        ax[r, i].imshow(m[i].cpu(), cmap='jet', alpha=0.5)
        ax[r, 0].set_ylabel(name)
for a in ax.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Preliminary faithfulness score (one model-method pair)

In [ ]:
from src.metrics import evaluate_explainer
loader = get_eval_subset(batch_size=16)
# max_batches=10 -> 160 images for a quick check; set None for the full 1000
res = evaluate_explainer(model, loader, gradcam, device, max_batches=10)
print('vit + Grad-CAM:', res)

## Full evaluation (writes/merges `results.json` on Drive)
Remove `--max-batches 4` for the full 1,000 images. Run the `--removal mean` line too (ablation).

In [ ]:
CK = f'{CKPT_DIR}/vit.pt'
!python -m src.run_eval --arch vit --ckpt {CK} --out $CKPT_DIR/results.json --max-batches 4
# !python -m src.run_eval --arch vit --ckpt {CK} --out $CKPT_DIR/results.json --removal mean

## Randomization sanity check (cascading, top-down)

In [ ]:
!python -m src.sanity --arch vit --ckpt {CK} --out $CKPT_DIR/sanity_vit.json --n-images 64